# 06.02 · Loading and parsing documents (PDF, HTML, Markdown)

**Module 06 · Documents, Embeddings & RAG** · Part 1 (Core)

| | |
|---|---|
| ⏱️ Estimated time | 90–120 min |
| 📋 Prerequisites | Lesson 06.01. |
| 🔑 Keys | none (no model calls in this lesson) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-12 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Load** PDFs page by page, HTML without boilerplate, and Markdown, into `Document(page_content, metadata)`.
2. **Keep** the metadata citations need: `source` (a stable, portable path), `page` (1-based), `title`, `type`.
3. **Inspect** what a loader really extracted, and spot scanned PDFs, broken line breaks and encoding problems.
4. **Decide** when a heavier parser (Docling, Unstructured) is worth it.

## 📚 Official reading (verified 2026-10-12)

| Page | Read it for |
|---|---|
| [Semantic search: loading documents](https://docs.langchain.com/oss/python/langchain/knowledge-base) | The tutorial now reads PDFs with `pypdf` directly and builds `Document`s with `source` and `page` |
| [Document loaders (integrations)](https://docs.langchain.com/oss/python/integrations/document_loaders) | The catalogue: PDFs, web pages, Office files, SaaS sources |
| [Docling loader](https://docs.langchain.com/oss/python/integrations/document_loaders/docling) | Layout-aware PDF/DOCX parsing (tables, reading order); heavy install |
| [Unstructured loader](https://docs.langchain.com/oss/python/integrations/document_loaders/unstructured_file) | Many formats, local or hosted API |
| [pypdf: extract text](https://pypdf.readthedocs.io/en/stable/user/extract-text.html) | What text extraction can and can't do |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain-core>=1.6.9,<2" "pypdf>=6.20.1,<7" "beautifulsoup4>=4.15.0,<5"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("langchain-core", "pypdf", "beautifulsoup4",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Step 1: Find the repository root (Jupyter starts in modules/06-rag/).
import importlib.util
import sys
from pathlib import Path

REPO_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone.")

# Step 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY',)
# Step 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")

In [ ]:
# Step 4: Silence known, harmless noise so real problems stand out. Set these BEFORE importing chromadb/fastembed.
import os
import warnings

os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")          # Chroma: no usage telemetry.
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")   # Windows: HF cache can't symlink without Developer Mode.
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")        # HF tokenizers' fork warning in Jupyter.
# Same as Lesson 02.04: langchain-core 1.6.9/1.6.10 Pydantic serializer warning about the injected runtime.
warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)
# tqdm (used by the model downloader) asks for ipywidgets to draw a progress bar; text output is fine.
warnings.filterwarnings("ignore", message="IProgress not found")

In [ ]:
# Step 5: The course corpus: 24 Markdown files + 8 PDFs (23 pages) about fictional Nimbus weather stations.
import json

RAG_DIR = REPO_ROOT / "data" / "samples" / "rag"
CORPUS_DIR = RAG_DIR / "corpus"
GOLDEN = [json.loads(line) for line in (RAG_DIR / "golden-set.jsonl").read_text(encoding="utf-8").splitlines()]
print("corpus files:", sum(1 for p in CORPUS_DIR.rglob("*") if p.is_file()), "| golden questions:", len(GOLDEN))

**You should now see** `corpus files: 32 | golden questions: 20`.

---
## Part 1 · What is in the corpus?

In [ ]:
# 🧪 Offline: list the corpus by folder and type.
from collections import Counter

files = sorted(p.relative_to(CORPUS_DIR).as_posix() for p in CORPUS_DIR.rglob("*") if p.is_file())
print(Counter(f.split("/")[0] for f in files))
print(Counter(f.rsplit(".", 1)[-1] for f in files))
print(files[:6], "...")

**You should now see** folders such as `products`, `support`, `policies`, `release-notes`, `manuals`, ... and `Counter({'md': 24, 'pdf': 8})`.

---
## Part 2 · A `Document` is text + metadata

Every loader returns `Document` objects. The **metadata** is what makes a citation possible later: without `source` and `page` you can't say *where* an answer came from.

In [ ]:
# 🧪 Offline: build a Document by hand, then load a Markdown file with the course loader.
from langchain_core.documents import Document

from mastery.rag import load_markdown

by_hand = Document(page_content="Returns are free within 30 days.", metadata={"source": "policies/returns.md"})
print(by_hand)

md_doc = load_markdown(CORPUS_DIR / "products" / "nimbus-pro.md", root=CORPUS_DIR)[0]
print(md_doc.metadata)
print(md_doc.page_content[:160], "...")

**You should now see** the hand-made `Document`, then `{'source': 'products/nimbus-pro.md', 'type': 'markdown', 'title': 'Nimbus Pro'}` and the start of the file.

`source` is **relative and uses forward slashes**, even on Windows. Absolute paths (`C:\\Users\\ana\\...`) leak your username into citations and break when the repo moves.

---
## Part 3 · PDFs: one `Document` per page

Users cite PDFs by page, so the loader makes **one `Document` per page** with a **1-based** `page` (what a PDF viewer shows). Careful: the LangChain tutorial stores pypdf's **0-based** index; pick one convention and stick to it. This course uses 1-based.

In [ ]:
# 🧪 Offline: load a 4-page manual with pypdf (via mastery.rag.load_pdf) and look at page 4.
from mastery.rag import load_pdf

pages = load_pdf(CORPUS_DIR / "manuals" / "nimbus-pro-manual.pdf", root=CORPUS_DIR)
print(len(pages), "pages")
print(pages[3].metadata)
print(repr(pages[3].page_content[:200]))

**You should now see** `4 pages`, metadata with `'page': 4, 'pages': 4`, and text that contains `Error E12 means the rain gauge is\nblocked`. Notice the `\n` **in the middle of a sentence**: PDF text keeps the visual line breaks. That matters twice: chunk boundaries (06.03) and quote checks (06.06 normalises whitespace before comparing).

Here is the raw pypdf call underneath, the same one the LangChain tutorial uses:

In [ ]:
# 🧪 Offline: pypdf directly.
from pypdf import PdfReader

reader = PdfReader(CORPUS_DIR / "plans" / "service-plans.pdf")
for number, page in enumerate(reader.pages, start=1):
    text = page.extract_text() or ""
    print(f"page {number}: {len(text)} chars | {text[:70]!r}")

**You should now see** two pages with their character counts. A page with **0 characters** is a red flag: it's probably a scanned image with no text layer. `load_pdf` warns and skips such pages; for real scans you need OCR (Docling or Unstructured can do it; see Part 6).

---
## Part 4 · HTML: keep the content, drop the chrome

Web pages carry navigation, scripts, cookie banners and footers. Indexed as-is, "Home | Status | Sign in" ends up in every chunk and pollutes retrieval.

In [ ]:
# 🧪 Offline: raw HTML text vs the cleaned loader.
from bs4 import BeautifulSoup

from mastery.rag import load_html

html_path = RAG_DIR / "html" / "status-page.html"
raw = BeautifulSoup(html_path.read_text(encoding="utf-8"), "html.parser").get_text(" ", strip=True)
print("RAW    :", raw[:220], "...")
clean = load_html(html_path, root=RAG_DIR)[0]
print("CLEAN  :", clean.metadata)
print(clean.page_content)

**You should now see** the raw text starting with `Orbita Status Home | Status | Sign in ...` (the page title and the navigation bar; the footer's cookie text is at the end), then the clean version: `# Service status`, the incident and the maintenance window, and no navigation, footer or cookie text. (Recent BeautifulSoup versions already skip `<script>`/`<style>` text in `get_text()`; older ones don't, which is one more reason to remove those tags explicitly.)

### What just happened?

`load_html` removes `script`, `style`, `nav`, `header`, `footer` and `form` tags, takes `<main>` when present, and keeps headings as Markdown (`#`, `##`) so the Markdown splitter in 06.03 can use them. For your own sites, check one page by eye: every site hides its content differently.

---
## Part 5 · Load the whole corpus

In [ ]:
# 🧪 Offline: everything, with a quick quality report.
from mastery.rag import load_corpus

docs = load_corpus(CORPUS_DIR)
print("documents:", len(docs), Counter(d.metadata["type"] for d in docs))
print("distinct sources:", len({d.metadata["source"] for d in docs}))
short = [d.metadata["source"] for d in docs if len(d.page_content.strip()) < 60]
print("suspiciously short:", short or "none")
print("non-ASCII characters:", sorted({ch for d in docs for ch in d.page_content if ord(ch) > 127}) or "none")

**You should now see** `documents: 47` (24 Markdown + 23 PDF pages), `distinct sources: 32`, `suspiciously short: none`, and `non-ASCII characters: ['°']` (from "°C" in a Markdown file: fine). Always run a report like this on a new corpus before indexing: empty pages, mojibake (`Ã©` instead of `é`) and duplicates are much cheaper to fix here than after a bad answer.

---
## Part 6 · Heavier parsers: Docling and Unstructured (mention only)

| Tool | Strengths | Cost | When |
|---|---|---|---|
| **pypdf** (this course) | Pure Python, small, fast, text PDFs | Free, ~1 MB | Born-digital PDFs with simple layout |
| **BeautifulSoup** (this course) | Any HTML, full control | Free | Web pages you can inspect |
| **[Docling](https://docs.langchain.com/oss/python/integrations/document_loaders/docling)** (`langchain-docling` 3.0.0, `docling` 2.138) | Layout analysis, tables, reading order, OCR, DOCX/PPTX | Free but **heavy**: PyTorch + models, GBs of disk, slow on CPU | Complex PDFs: multi-column, tables, scans |
| **[Unstructured](https://docs.langchain.com/oss/python/integrations/document_loaders/unstructured_file)** (`langchain-unstructured` 1.0.1) | Dozens of formats, element types (Title, Table...) | Local install is heavy (system libraries, OCR); hosted API is paid beyond the free tier | Mixed file types at scale |

We don't install them: our corpus doesn't need them, and on Windows their system dependencies (Poppler, Tesseract) are a project of their own. Rule of thumb: start with pypdf; if a page's text looks wrong in Part 3's check, try Docling on **that** document.

---
## ⚠️ Pitfalls and troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| A PDF page has empty text | Scanned image, no text layer | OCR (Docling/Unstructured, or OCR the PDF first); `load_pdf` warns and skips |
| Words glued together or split (`rain gauge is\nblocked`) | PDF layout | Normalise whitespace for matching; don't fight it in the loader |
| `UnicodeDecodeError: 'charmap' codec` on Windows | `open()` without encoding uses cp1252 | Always `read_text(encoding="utf-8")` (the course loaders do) |
| `\r` at line ends, Markdown headings not detected | CRLF files from a Windows editor or git `autocrlf` | `load_markdown` converts CRLF to LF |
| Citations show `C:\Users\...` | Absolute `source` | Relative POSIX paths (`root=`), as here |
| Page numbers off by one | 0-based vs 1-based | Choose one (we use 1-based) and document it |
| `ModuleNotFoundError: pypdf` / `bs4` | `[rag]` extra not installed | `python -m pip install -e ".[dev]"` (Before-you-start block) |

## ⚠️ Production notes

> ⚠️ **Production note: know what you index.**
> Private documents in an index are as sensitive as the documents themselves: control access per user (metadata filters, 06.05), keep personal data out unless you need it, and never commit real documents to a public repo. Our corpus is fictional on purpose.

> ⚠️ **Production note: loaders are an attack surface.**
> PDFs and HTML from outside can contain hidden text (white-on-white, tiny fonts) written for the model. Log what was extracted, and treat it as untrusted (06.06).

## 🏋️ Exercises

1. **(Easy)** Load `compliance/safety-and-compliance.pdf`. How many pages, and which page mentions lightning?
2. **(Medium)** Write a function that returns every `source` whose documents contain a given word (e.g. "warranty"). Which files mention it?
3. **(Medium)** Save a copy of a Markdown file with Windows line endings (`text.replace("\n", "\r\n")`, written with `newline=""`) into a temp folder and load it. Is there any `\r` left?
4. **(Stretch)** Create an HTML file without `<main>` and with a cookie banner in a `<div class="cookie">`. What does `load_html` keep? How would you remove the banner?

<details>
<summary><b>Show solutions</b></summary>

**1.** `pages = load_pdf(CORPUS_DIR / "compliance/safety-and-compliance.pdf", root=CORPUS_DIR)`: 3 pages; `[d.metadata["page"] for d in pages if "lightning" in d.page_content.lower()]`.

**2.** `sorted({d.metadata["source"] for d in docs if "warranty" in d.page_content.lower()})`: product pages, `policies/warranty.md` and others.

**3.** No: `load_markdown` replaces `\r\n` with `\n`. `import tempfile, pathlib; p = pathlib.Path(tempfile.mkdtemp()) / "x.md"; p.write_text(text.replace("\n", "\r\n"), encoding="utf-8", newline="")`.

**4.** Without `<main>` it uses `<body>`, so the banner's text stays. Remove it before extracting: `for el in soup.select(".cookie"): el.decompose()` (copy `load_html` and add the selector).
</details>

## ✅ Checkpoint

- [ ] I can load PDF pages (1-based `page`), clean HTML and Markdown into `Document`s.
- [ ] My `source` values are relative POSIX paths.
- [ ] I ran a quality report (counts, empty pages, odd characters) before indexing.
- [ ] I know when Docling or Unstructured would be worth the install.
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Research checkpoint

The LangChain knowledge-base tutorial switched from `PyPDFLoader` (langchain-community) to plain `pypdf`. Is that still the case, and which page number convention does it use? Check the [knowledge base](https://docs.langchain.com/oss/python/langchain/knowledge-base) page and log the answer in [`docs/research-log.md`](../../docs/research-log.md).

## Summary

- A `Document` = text + metadata; `source` and `page` make citations possible.
- PDFs: one Document per page (1-based), expect odd line breaks and empty scanned pages.
- HTML: remove navigation, scripts and footers before indexing.
- Check what you extracted before you index; heavy parsers only when a document needs them.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 06.03 · Chunking strategies** → [`06-03-chunking.ipynb`](06-03-chunking.ipynb).

[Module 06 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)